# convert peak_locations.pkl to mat

In [1]:
from scipy import io
import pickle

import nescafe23 as n23

In [6]:
peak_locations_full = n23.open_peak_locations()
peak_locations_QC = peak_locations_full.copy()

In [7]:
import numpy as np

# apply peak QC: keep flags 1 and 5, set everything else to NaN
# (a short/empty QC row is unreviewed, so the whole row is masked)
GOOD_QC = (1, 5)
qc_tables = {
    "": n23.PEAK_LOWER_QC,
    "upper_": n23.PEAK_UPPER_QC,
}

for prefix, qc_table in qc_tables.items():
    for var in ("times", "angles", "snrs"):
        for ray_id, values in peak_locations_QC[prefix + var].items():
            values = np.asarray(values, dtype=float)
            qc = np.asarray(qc_table.get(ray_id, []))
            good = np.zeros(values.shape, dtype=bool)
            if qc.shape == values.shape:
                good = np.isin(qc, GOOD_QC)
            peak_locations_QC[prefix + var][ray_id] = np.where(good, values, np.nan)

In [4]:
def sanitize_keys(obj):
    """Recursively sanitize dict keys to be valid MATLAB field names."""
    if isinstance(obj, dict):
        new = {}
        for k, v in obj.items():
            # Replace +/- and leading digits; strip other bad chars
            new_k = str(k).replace('+', 'p').replace('-', 'm')
            if new_k[0].isdigit():
                new_k = 'f' + new_k
            new[new_k] = sanitize_keys(v)
        return new
    elif isinstance(obj, list):
        return [sanitize_keys(i) for i in obj]
    else:
        return obj

In [12]:
io.savemat('peak_locations.mat', {'peak_locations_QC': sanitize_keys(peak_locations_QC), 'peak_locations_full': sanitize_keys(peak_locations_full)})

In [15]:
import nescafe23 as n23

In [16]:
dss = n23.open_receptions()

100%|██████████| 23/23 [00:21<00:00,  1.09it/s]


In [18]:
dss[0].nominal_depth

<xarray.DataArray 'nominal_depth' (hydrophone: 60)> Size: 480B
array([  88.,   91.,   94.,   97.,  100.,  109.,  118.,  127.,  136.,
        145.,  154.,  163.,  172.,  181.,  190.,  199.,  208.,  217.,
        226.,  235.,  244.,  253.,  262.,  271.,  280.,  289.,  298.,
        307.,  316.,  325.,  334.,  343.,  850.,  859.,  868.,  877.,
        886.,  895.,  904.,  913.,  922.,  931.,  940.,  949.,  958.,
        967.,  976.,  985.,  994., 1003., 1012., 1021., 1030., 1039.,
       1048., 1057., 1066., 1075., 1084., 1093.])
Coordinates:
    nominal_depth  (hydrophone) float64 480B 88.0 91.0 ... 1.084e+03 1.093e+03
    actual_depth   (hydrophone) float64 480B 172.2 174.7 ... 1.19e+03 1.203e+03
Dimensions without coordinates: hydrophone